### Notebook to prepare dataset and to create metadata

* For simplicity, only the val2017 will be used in this project

In [8]:
import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/coco")
IMAGE_DIR = DATA_DIR / "images" / "val2017"
ANN_FILE = DATA_DIR / "annotations" / "instances_val2017.json"

# annotations
with open(ANN_FILE, "r") as f:
    coco = json.load(f)

# categories
categories = pd.DataFrame(coco["categories"])
print(f"number of categories: {len(categories)}")
print(categories.to_string())

number of categories: 80
   supercategory  id            name
0         person   1          person
1        vehicle   2         bicycle
2        vehicle   3             car
3        vehicle   4      motorcycle
4        vehicle   5        airplane
5        vehicle   6             bus
6        vehicle   7           train
7        vehicle   8           truck
8        vehicle   9            boat
9        outdoor  10   traffic light
10       outdoor  11    fire hydrant
11       outdoor  13       stop sign
12       outdoor  14   parking meter
13       outdoor  15           bench
14        animal  16            bird
15        animal  17             cat
16        animal  18             dog
17        animal  19           horse
18        animal  20           sheep
19        animal  21             cow
20        animal  22        elephant
21        animal  23            bear
22        animal  24           zebra
23        animal  25         giraffe
24     accessory  27        backpack
25     access

In [13]:
selected_categories = ['person', 'car', 'truck', 'dog', 'cat']
selected_cats = categories[categories['name'].isin(selected_categories)]
cat_id_to_name = dict(zip(selected_cats['id'], selected_cats['name']))
print(cat_id_to_name)

{1: 'person', 3: 'car', 8: 'truck', 17: 'cat', 18: 'dog'}


In [16]:
annotations_df = pd.DataFrame(coco['annotations'])
images_df = pd.DataFrame(coco['images'])

In [18]:
# filter annotations for selected categories
filtered_annotations = annotations_df[annotations_df['category_id'].isin(cat_id_to_name.keys())]

In [22]:
selected_image_ids = filtered_annotations['image_id'].unique()

In [ ]:
# filter image
selected_images = images_df[images_df['id'].isin(selected_image_ids)].copy()
selected_images['file_path'] = selected_images['file_name'].apply(lambda x: str(IMAGE_DIR / x))

In [25]:
print(f'Total images: {len(selected_images)}')
print(selected_images[['id', 'file_name', 'file_path']].head())

Total images: 3141
       id         file_name                                     file_path
0  397133  000000397133.jpg  ..\data\coco\images\val2017\000000397133.jpg
2  252219  000000252219.jpg  ..\data\coco\images\val2017\000000252219.jpg
3   87038  000000087038.jpg  ..\data\coco\images\val2017\000000087038.jpg
4  174482  000000174482.jpg  ..\data\coco\images\val2017\000000174482.jpg
7  480985  000000480985.jpg  ..\data\coco\images\val2017\000000480985.jpg


In [26]:
selected_images = selected_images[['id', 'file_name', 'file_path']].reset_index(drop=True)

In [32]:
img_to_cats = filtered_annotations.groupby('image_id')['category_id'].apply(list).reset_index()
img_to_cats.columns = ['id', 'category_ids']
img_to_cats['labels'] = img_to_cats['category_ids'].apply(lambda ids: list(set([cat_id_to_name[id] for id in ids])))

In [33]:
img_to_cats

,id,category_ids,labels
0,139,"[1, 1]",[person]
1,724,"[8, 3]","[car, truck]"
2,785,[1],[person]
3,872,"[1, 1]",[person]
4,885,"[1, 1, 1, 1, 1, 1, 1, 1]",[person]
...,...,...,...
3136,580418,"[3, 3, 3, 1]","[car, person]"
3137,581062,[1],[person]
3138,581206,[1],[person]
3139,581317,[1],[person]


In [ ]:
# merge labels
selected_images = selected_images.merge(img_to_cats[['id', 'category_ids', 'labels']], on='id')

In [40]:
selected_images.head()

,id,file_name,file_path,category_ids,labels
0,397133,000000397133.jpg,..\data\coco\images\val2017\000000397133.jpg,"[1, 1]",[person]
1,252219,000000252219.jpg,..\data\coco\images\val2017\000000252219.jpg,"[1, 1, 1]",[person]
2,87038,000000087038.jpg,..\data\coco\images\val2017\000000087038.jpg,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",[person]
3,174482,000000174482.jpg,..\data\coco\images\val2017\000000174482.jpg,"[3, 3, 3, 3, 3, 8, 8, 8]","[truck, car]"
4,480985,000000480985.jpg,..\data\coco\images\val2017\000000480985.jpg,"[1, 1, 1, 1, 1, 1, 1, 1]",[person]


In [38]:
selected_images.shape

(3141, 5)

In [41]:
print(f'Total images: {len(selected_images)}')
print(selected_images.head())

Total images: 3141
       id         file_name                                     file_path  \
0  397133  000000397133.jpg  ..\data\coco\images\val2017\000000397133.jpg   
1  252219  000000252219.jpg  ..\data\coco\images\val2017\000000252219.jpg   
2   87038  000000087038.jpg  ..\data\coco\images\val2017\000000087038.jpg   
3  174482  000000174482.jpg  ..\data\coco\images\val2017\000000174482.jpg   
4  480985  000000480985.jpg  ..\data\coco\images\val2017\000000480985.jpg   

                                 category_ids        labels  
0                                      [1, 1]      [person]  
1                                   [1, 1, 1]      [person]  
2  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]      [person]  
3                    [3, 3, 3, 3, 3, 8, 8, 8]  [truck, car]  
4                    [1, 1, 1, 1, 1, 1, 1, 1]      [person]  


In [42]:
selected_images['labels'] = selected_images['labels'].apply(json.dumps)

In [44]:
selected_images.to_csv('../data/coco/metadata.csv', index=False)

In [47]:
print(selected_images['labels'].value_counts().head(10))

labels
["person"]                    2195
["car", "person"]              249
["cat"]                        143
["car"]                        111
["truck", "car", "person"]      91
["dog"]                         79
["dog", "person"]               62
["truck", "person"]             54
["truck", "car"]                41
["truck"]                       38
Name: count, dtype: int64
